# Multi-Frame Super-Resolution with Optotune XPR-20 Beam Shifting

**Project:** ENPH 459 — Super-Resolution Imaging  
**Hardware:** Daheng MER2-302-56U3C (3.45 µm pitch) + Kowa LM35HC-OPT + Optotune XPR-20-4P  
**Target:** ISO 12233 resolution chart (centre region)

---

## How this notebook is organised

| Section | What it does |
|---------|-------------|
| **0. Setup** | Imports, configuration, file paths — the only cell you need to edit |
| **1. Data Loading** | Load images, visualise, estimate sub-pixel shifts |
| **2. PSF Kernel** | Load the measured system PSF for use in the forward model |
| **3. Super-Resolution** | Three methods of increasing sophistication |
| 3a. Shift-and-Add (SAA) | Simplest baseline — place LR frames on HR grid and average |
| 3b. SAA + Iterative Back-Projection (IBP) | Iteratively correct SAA using the known PSF |
| 3c. Total Variation SR (TV-SR) | Optimisation-based SR with edge-preserving regularisation |
| **4. Resolution Measurement** | Four complementary ways to quantify improvement |
| 4a. Slanted-edge MTF | Gold-standard frequency-domain resolution metric |
| 4b. Siemens star contrast | Contrast vs spatial frequency from the concentric rings |
| 4c. Line-pair profiles | Visual comparison of fine-feature contrast |
| 4d. Ground-truth comparison | SIFT + ECC registration against the chart PDF → PSNR / SSIM |
| **5. Summary** | Side-by-side dashboard of all results |

---
## 0. Setup

**This is the only cell you should need to edit.** Everything else flows from these settings.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CONFIGURATION — edit these to match your local setup       ║
# ╚══════════════════════════════════════════════════════════════╝

# --- File paths ---
IMAGE_DIR      = 'imaging/calibration'
CENTER_PATH    = f'{IMAGE_DIR}/center.png'           # XPR off (reference)
SHIFT_PATHS    = [f'{IMAGE_DIR}/shift_{i}.png' for i in range(4)]  # 4 XPR positions
PSF_DATA_PATH  = f'{IMAGE_DIR}/psf_for_sr_data.npz'  # output of psf_mtf.py
ISO_CHART_PATH = f'{IMAGE_DIR}/ISO_12233-reschart.pdf'

# --- Sensor ---
PIXEL_PITCH_UM = 3.45   # µm per pixel

# --- Sub-pixel shifts ---
# (dy, dx) in LR pixels for each of the 4 XPR-20 beam positions
# relative to the center (reference) image.
# The XPR-20 4P mode produces a 2×2 grid at ±half-pixel offsets.
# If your SR result looks blurry or ghosted, try permuting these.
NOMINAL_SHIFTS = [
    (-0.5, -0.5),   # shift_0 → position A (top-left)
    (+0.5, -0.5),   # shift_1 → position B (bottom-left)
    (-0.5, +0.5),   # shift_2 → position C (top-right)
    (+0.5, +0.5),   # shift_3 → position D (bottom-right)
]
USE_MEASURED_SHIFTS = False  # True → use phase-correlation estimates instead

# --- SR parameters (tune these in Parts 3b and 3c) ---
UPSAMPLE_FACTOR = 2
IBP_ITERATIONS  = 30
IBP_STEP_SIZE   = 0.5
TV_ITERATIONS   = 60
TV_STEP_SIZE    = 0.15
TV_LAMBDA       = 0.005

# --- ROI coordinates (in LR pixels — adjust after viewing Part 1) ---
# Slanted edge for MTF measurement
EDGE_ROI_LR     = (550, 750, 580, 620)  # (r0, r1, c0, c1)
EDGE_ORIENT     = 'vertical'
# Siemens star / concentric rings centre
STAR_CENTER_LR  = (750, 1000)            # (row, col)
# Line-pair profile
LP_ROW_LR       = 950
LP_COL_RANGE_LR = (100, 500)

In [ ]:
# --- Imports (nothing to edit here) ---
import os, subprocess, warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from PIL import Image
from scipy.ndimage import shift as ndi_shift, gaussian_filter, zoom as ndi_zoom
from scipy.signal import convolve2d
from skimage.registration import phase_cross_correlation
from skimage.metrics import structural_similarity as ssim, peak_signal_noise_ratio as psnr
import cv2

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 8)})

# Derived constants
f = UPSAMPLE_FACTOR
PITCH_LR_MM = PIXEL_PITCH_UM * 1e-3
PITCH_HR_MM = PITCH_LR_MM / f
NYQUIST_LR  = 0.5 / PITCH_LR_MM   # cycles/mm
NYQUIST_HR  = 0.5 / PITCH_HR_MM

COLORS = {'Native': 'k', 'Bicubic 2×': 'gray',
          'SAA': 'C0', 'SAA + IBP': 'C2', 'TV-SR': 'C3'}

def load_gray(path):
    """Load any image as float64 grayscale."""
    img = np.array(Image.open(path), dtype=np.float64)
    return img.mean(axis=2) if img.ndim == 3 else img

print('Setup complete.')
print(f'  LR Nyquist:  {NYQUIST_LR:.1f} cy/mm')
print(f'  HR Nyquist:  {NYQUIST_HR:.1f} cy/mm  ({f}× upsample)')

---
## 1. Data Loading & Shift Estimation

We have 5 images of the ISO 12233 chart centre:
- **`center.png`** — captured with the XPR-20 off (our native-resolution reference)
- **`shift_0.png` … `shift_3.png`** — captured at the 4 beam-shifted positions

The XPR-20 tilts a glass plate to translate the optical beam by ~half a pixel
in a 2×2 grid, giving us 4 sub-pixel-offset views of the same scene.
Combining these is the basis for 2× super-resolution.

In [ ]:
# Load all images
lr_ref = load_gray(CENTER_PATH)
lr_images = [load_gray(p) for p in SHIFT_PATHS]

print(f'Reference image : {lr_ref.shape}  (center, XPR off)')
print(f'Shifted images  : {len(lr_images)} × {lr_images[0].shape}')
print(f'Dynamic range   : [{lr_ref.min():.0f}, {lr_ref.max():.0f}]')

In [ ]:
# Visualise all 5 images
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
for ax, img, title in zip(axes,
                           [lr_ref] + lr_images,
                           ['center (ref)'] + [f'shift_{i}' for i in range(4)]):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.suptitle('Raw captures', fontsize=12)
plt.tight_layout(); plt.show()

### 1.1 Sub-pixel shift estimation

We use **upsampled phase cross-correlation** (from scikit-image) to measure the
actual sub-pixel displacement of each shifted image relative to the centre.

Since the shifts are only ~0.5 px, they sit right at the aliasing boundary and
phase correlation can struggle. If the measured values look wrong, use the
nominal shifts instead (`USE_MEASURED_SHIFTS = False` in the config).

In [ ]:
measured_shifts = []
print(f'{"Image":>8s}   {"Measured (dy, dx)":>20s}   {"Nominal (dy, dx)":>20s}')
print('-' * 55)

for i, img in enumerate(lr_images):
    shift_est, _, _ = phase_cross_correlation(lr_ref, img, upsample_factor=200)
    measured_shifts.append(tuple(shift_est))
    nom = NOMINAL_SHIFTS[i]
    print(f'shift_{i}   ({shift_est[0]:+7.4f}, {shift_est[1]:+7.4f})'
          f'   ({nom[0]:+5.2f}, {nom[1]:+5.2f})')

sub_shifts = measured_shifts if USE_MEASURED_SHIFTS else NOMINAL_SHIFTS
print(f'\n>>> Using {"MEASURED" if USE_MEASURED_SHIFTS else "NOMINAL"} shifts')

In [ ]:
# Plot the shift pattern
fig, ax = plt.subplots(figsize=(4.5, 4.5))
for i, (dy, dx) in enumerate(sub_shifts):
    ax.plot(dx, dy, 'o', markersize=14, zorder=3,
            label=f'shift_{i} ({dy:+.2f}, {dx:+.2f})')
ax.plot(0, 0, 'kx', markersize=14, mew=3, zorder=4, label='center (0, 0)')
ax.set_xlabel('Δx (pixels)'); ax.set_ylabel('Δy (pixels)')
ax.set_title('XPR-20 shift pattern')
ax.legend(fontsize=8, loc='upper left')
ax.set_aspect('equal')
ax.set_xlim(-1.0, 1.0); ax.set_ylim(-1.0, 1.0)
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 2. System PSF Kernel

The IBP and TV-SR methods need a **forward model** that simulates how an HR scene
becomes a LR observation:

$$\mathbf{y}_k = D \cdot S_k \cdot H \cdot \mathbf{x} + \mathbf{n}$$

where $\mathbf{x}$ is the HR image, $H$ is blur (PSF convolution), $S_k$ is the
sub-pixel shift for frame $k$, $D$ is decimation, and $\mathbf{n}$ is noise.

We load the PSF measured from the backlit-aperture experiment (`psf_mtf.py`).
If that file isn't available, we fall back to a Gaussian approximation.

In [ ]:
PSF_HALFWIDTH = 3  # extract a (2*hw+1) × (2*hw+1) kernel → 7×7

try:
    psf_data = np.load(PSF_DATA_PATH, allow_pickle=True)
    psf_full = psf_data['psf_avg']
    cy, cx = psf_full.shape[0] // 2, psf_full.shape[1] // 2
    psf_kernel = psf_full[cy - PSF_HALFWIDTH : cy + PSF_HALFWIDTH + 1,
                          cx - PSF_HALFWIDTH : cx + PSF_HALFWIDTH + 1].copy()
    psf_kernel = np.clip(psf_kernel, 0, None)
    psf_kernel /= psf_kernel.sum()
    psf_source = 'measured (pinhole)'
except FileNotFoundError:
    sigma = 0.8  # approximate from earlier Gaussian fit
    k = 2 * PSF_HALFWIDTH + 1
    ax_k = np.arange(k) - k // 2
    xx, yy = np.meshgrid(ax_k, ax_k)
    psf_kernel = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    psf_kernel /= psf_kernel.sum()
    psf_source = f'Gaussian (σ = {sigma:.1f} px)'

# The SR forward model operates at HR resolution (2× finer pixels),
# so we upsample the LR PSF kernel to the HR grid.
psf_hr = ndi_zoom(psf_kernel, f, order=3)
psf_hr = np.clip(psf_hr, 0, None)
psf_hr /= psf_hr.sum()

print(f'PSF source    : {psf_source}')
print(f'LR kernel     : {psf_kernel.shape}')
print(f'HR kernel (2×): {psf_hr.shape}')

fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(psf_kernel, cmap='inferno', interpolation='nearest')
axes[0].set_title(f'PSF kernel ({psf_kernel.shape[0]}×{psf_kernel.shape[1]})')
axes[1].imshow(np.log10(psf_kernel + 1e-8), cmap='inferno', interpolation='nearest')
axes[1].set_title('Log scale')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

---
## 3. Super-Resolution Methods

All three methods share a common **forward model** that simulates how an HR
estimate would look if captured as a LR image at a given shift position.

In [ ]:
# ── Forward / adjoint operators ────────────────────────────────

def blur(img, kernel):
    """Convolve image with a PSF kernel (symmetric boundary)."""
    return convolve2d(img, kernel, mode='same', boundary='symm')

def forward_model(hr, kernel, shift_yx, factor):
    """
    Simulate a LR observation from an HR image.
    
    Pipeline: HR → blur(PSF) → sub-pixel shift → decimate
    
    Parameters
    ----------
    hr        : 2D HR image
    kernel    : PSF at HR resolution
    shift_yx  : (dy, dx) shift in LR pixels
    factor    : decimation factor (2 for 2× SR)
    """
    blurred = blur(hr, kernel)
    shifted = ndi_shift(blurred,
                        (shift_yx[0] * factor, shift_yx[1] * factor),
                        order=3, mode='nearest')
    return shifted[::factor, ::factor]  # decimate

def back_project(error_lr, kernel, shift_yx, factor, hr_shape):
    """
    Adjoint of the forward model: project a LR error back onto the HR grid.
    
    Pipeline: upsample (zero-insert) → reverse shift → correlate with PSF
    """
    h_hr, w_hr = hr_shape
    # Zero-insert upsample
    up = np.zeros((error_lr.shape[0] * factor, error_lr.shape[1] * factor))
    up[::factor, ::factor] = error_lr
    # Crop / pad to HR size
    up = up[:h_hr, :w_hr] if up.shape[0] >= h_hr else \
         np.pad(up, ((0, h_hr - up.shape[0]), (0, w_hr - up.shape[1])))
    # Reverse shift
    shifted = ndi_shift(up,
                        (-shift_yx[0] * factor, -shift_yx[1] * factor),
                        order=3, mode='nearest')
    # Correlation with PSF = convolution with flipped kernel (adjoint)
    return blur(shifted, kernel[::-1, ::-1])

print('Forward model defined.')

### 3a. Shift-and-Add (SAA)

The simplest multi-frame SR approach:
1. Upsample each LR image to the HR grid (nearest-neighbour repeat)
2. Shift it by the known sub-pixel offset
3. Average all contributions

This recovers some aliased information but does **not** deconvolve the PSF,
so the result is still blurry. It serves as the initialisation for IBP and TV-SR.

In [ ]:
def shift_and_add(lr_list, shifts_yx, factor=2):
    """Shift-and-Add: upsample each LR frame, shift, and average."""
    h_lr, w_lr = lr_list[0].shape
    accumulator = np.zeros((h_lr * factor, w_lr * factor))
    for lr, (dy, dx) in zip(lr_list, shifts_yx):
        upsampled = np.repeat(np.repeat(lr, factor, axis=0), factor, axis=1)
        shifted = ndi_shift(upsampled,
                            (dy * factor, dx * factor),
                            order=3, mode='nearest')
        accumulator += shifted
    return accumulator / len(lr_list)

sr_saa = shift_and_add(lr_images, sub_shifts, f)
print(f'SAA result: {sr_saa.shape}')

In [ ]:
# Also make a bicubic baseline (just upscaling the center image, no SR)
baseline_bicubic = ndi_zoom(lr_ref, f, order=3)

# Quick side-by-side
cr, cc = lr_ref.shape[0] // 2, lr_ref.shape[1] // 2
crop_r = 70  # LR pixels

def hr_crop(img, cr, cc, crop_r, factor):
    """Extract a crop from an HR image using LR coordinates."""
    return img[(cr-crop_r)*factor:(cr+crop_r)*factor,
               (cc-crop_r)*factor:(cc+crop_r)*factor]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, title in zip(axes,
    [ndi_zoom(lr_ref, f, order=0), baseline_bicubic, sr_saa],
    ['Native (nearest 2×)', 'Bicubic 2×', 'SAA 2×']):
    ax.imshow(hr_crop(img, cr, cc, crop_r, f), cmap='gray', interpolation='nearest')
    ax.set_title(title); ax.axis('off')
plt.suptitle('SAA vs baselines (centre crop)', fontsize=12)
plt.tight_layout(); plt.show()

### 3b. SAA + Iterative Back-Projection (IBP)

IBP refines the SAA estimate by repeatedly:
1. **Forward-projecting** the current HR estimate through the known PSF + shift + decimate to simulate each LR frame
2. **Computing the error** between simulated and actual LR frames
3. **Back-projecting** the error onto the HR grid (adjoint of the forward model)
4. **Updating** the HR estimate with a weighted correction

This effectively deconvolves the PSF while enforcing consistency with all
observed LR frames. More iterations → sharper, but too many → ringing artefacts.

**Key parameters:**
- `IBP_ITERATIONS`: 20–50 is typical. Watch the convergence curve.
- `IBP_STEP_SIZE`: 0.3–0.8. Smaller = stabler but slower.

In [ ]:
def ibp(lr_list, shifts_yx, kernel, hr_init, factor=2,
        n_iter=20, step=0.5, verbose=True):
    """
    Iterative Back-Projection super-resolution.
    
    Returns
    -------
    hr     : final HR estimate
    errors : per-iteration MSE (for convergence monitoring)
    """
    hr = hr_init.copy()
    n = len(lr_list)
    errors = []

    for it in range(n_iter):
        correction = np.zeros_like(hr)
        total_err = 0.0

        for lr, s in zip(lr_list, shifts_yx):
            sim = forward_model(hr, kernel, s, factor)
            mh, mw = min(sim.shape[0], lr.shape[0]), min(sim.shape[1], lr.shape[1])
            err = lr[:mh, :mw] - sim[:mh, :mw]
            total_err += np.mean(err ** 2)
            correction += back_project(err, kernel, s, factor, hr.shape)

        hr += step * correction / n
        hr = np.clip(hr, 0, 255)
        errors.append(total_err / n)

        if verbose and (it % 5 == 0 or it == n_iter - 1):
            print(f'  iter {it:3d}/{n_iter}  MSE = {errors[-1]:.3f}')

    return hr, errors

In [ ]:
print(f'Running IBP  (iterations={IBP_ITERATIONS}, step={IBP_STEP_SIZE}) ...')
sr_ibp, ibp_errors = ibp(
    lr_images, sub_shifts, psf_hr, sr_saa.copy(),
    factor=f, n_iter=IBP_ITERATIONS, step=IBP_STEP_SIZE
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(ibp_errors, 'C2-', lw=1.5)
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('MSE')
axes[0].set_title('IBP convergence'); axes[0].grid(True, alpha=0.3)

axes[1].imshow(hr_crop(sr_ibp, cr, cc, crop_r, f), cmap='gray', interpolation='nearest')
axes[1].set_title('IBP result (centre crop)'); axes[1].axis('off')
plt.tight_layout(); plt.show()

### 3c. Total Variation Regularised SR (TV-SR)

TV-SR formulates super-resolution as an optimisation problem:

$$\min_{\mathbf{x}} \; \frac{1}{K} \sum_{k=1}^{K} \| D S_k H \mathbf{x} - \mathbf{y}_k \|^2
  \;+\; \lambda \cdot \mathrm{TV}(\mathbf{x})$$

- **Data term**: each LR image should match the forward-projected HR estimate
- **TV term**: penalises noisy oscillations while preserving sharp edges

Solved by gradient descent. The TV regulariser prevents the noise amplification
that IBP can suffer from, at the cost of slightly softer textures.

**Key parameters:**
- `TV_LAMBDA`: regularisation weight. Larger → smoother. Try 0.001 – 0.05.
- `TV_ITERATIONS`: 40–100. Check the convergence curve.
- `TV_STEP_SIZE`: 0.05–0.3. Too large → oscillates.

In [ ]:
def tv_gradient(img):
    """Gradient of isotropic Total Variation (sub-differential)."""
    eps = 1e-8
    dy = np.zeros_like(img); dy[:-1, :] = img[1:, :] - img[:-1, :]
    dx = np.zeros_like(img); dx[:, :-1] = img[:, 1:] - img[:, :-1]
    mag = np.sqrt(dy**2 + dx**2 + eps)
    ny, nx = dy / mag, dx / mag
    div = np.zeros_like(img)
    div[1:, :]  -= ny[:-1, :]; div[:-1, :] += ny[:-1, :]
    div[:, 1:]  -= nx[:, :-1]; div[:, :-1] += nx[:, :-1]
    return -div


def tv_sr(lr_list, shifts_yx, kernel, hr_init, factor=2,
          n_iter=50, step=0.1, lam=0.01, verbose=True):
    """
    Total Variation regularised super-resolution via gradient descent.
    
    Returns
    -------
    hr     : final HR estimate
    errors : per-iteration data-fidelity MSE
    """
    hr = hr_init.copy()
    n = len(lr_list)
    errors = []

    for it in range(n_iter):
        data_grad = np.zeros_like(hr)
        total_err = 0.0

        for lr, s in zip(lr_list, shifts_yx):
            sim = forward_model(hr, kernel, s, factor)
            mh, mw = min(sim.shape[0], lr.shape[0]), min(sim.shape[1], lr.shape[1])
            residual = sim[:mh, :mw] - lr[:mh, :mw]
            total_err += np.mean(residual ** 2)
            data_grad += back_project(residual, kernel, s, factor, hr.shape)

        data_grad /= n
        hr -= step * (data_grad + lam * tv_gradient(hr))
        hr = np.clip(hr, 0, 255)
        errors.append(total_err / n)

        if verbose and (it % 10 == 0 or it == n_iter - 1):
            print(f'  iter {it:3d}/{n_iter}  MSE = {errors[-1]:.3f}')

    return hr, errors

In [ ]:
print(f'Running TV-SR  (iterations={TV_ITERATIONS}, step={TV_STEP_SIZE}, λ={TV_LAMBDA}) ...')
sr_tv, tv_errors = tv_sr(
    lr_images, sub_shifts, psf_hr, sr_saa.copy(),
    factor=f, n_iter=TV_ITERATIONS, step=TV_STEP_SIZE, lam=TV_LAMBDA
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(tv_errors, 'C3-', lw=1.5)
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('MSE')
axes[0].set_title('TV-SR convergence'); axes[0].grid(True, alpha=0.3)

axes[1].imshow(hr_crop(sr_tv, cr, cc, crop_r, f), cmap='gray', interpolation='nearest')
axes[1].set_title('TV-SR result (centre crop)'); axes[1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# ── Collect all results for comparison ──────────────────────────
results = {
    'Native':     ndi_zoom(lr_ref, f, order=0),  # nearest-neighbour (no new info)
    'Bicubic 2×': baseline_bicubic,               # interpolation only
    'SAA':        sr_saa,
    'SAA + IBP':  sr_ibp,
    'TV-SR':      sr_tv,
}

# Side-by-side
fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
for ax, (name, img) in zip(axes, results.items()):
    ax.imshow(hr_crop(img, cr, cc, 55, f), cmap='gray', interpolation='nearest')
    ax.set_title(name, fontsize=10); ax.axis('off')
plt.suptitle('All methods — centre of ISO 12233 chart', fontsize=13)
plt.tight_layout(); plt.show()

---
## 4. Resolution Measurement

We use four complementary approaches, each answering a slightly different question:

| Method | What it measures | Strengths |
|--------|-----------------|------------|
| Slanted-edge MTF | Contrast transfer vs frequency | Gold-standard, quantitative |
| Siemens star contrast | Contrast vs radius (≈ frequency) | Uses the chart's built-in pattern |
| Line-pair profiles | Visual contrast of fine features | Intuitive, easy to inspect |
| Ground-truth comparison | PSNR / SSIM vs known chart | Captures reconstruction fidelity |

### 4a. Slanted-Edge MTF

The standard method from ISO 12233:
1. Select a near-vertical (or horizontal) **high-contrast edge**
2. Fit the edge angle, then project all pixels onto the edge-normal direction  
   → this gives an **oversampled Edge Spread Function (ESF)** even from a single image
3. Differentiate the ESF → **Line Spread Function (LSF)**
4. FFT of the LSF → **MTF**

Compare the MTF curves from the native image vs each SR result.  
**MTF50** (frequency where MTF = 0.5) is the single-number resolution metric.

In [ ]:
def slanted_edge_mtf(img, roi, orientation='vertical', oversample=4):
    """
    Measure MTF from a slanted edge in the given ROI.
    
    Parameters
    ----------
    roi : (r0, r1, c0, c1)
    orientation : 'vertical' or 'horizontal'
    
    Returns
    -------
    freq_cpp : frequency in cycles/pixel
    mtf      : normalised MTF (1.0 at DC)
    esf      : oversampled Edge Spread Function
    lsf      : Line Spread Function
    """
    r0, r1, c0, c1 = roi
    patch = img[r0:r1, c0:c1].copy()
    if orientation == 'horizontal':
        patch = patch.T

    rows, cols = patch.shape

    # --- Locate edge per row via derivative centroid ---
    edge_pts = []
    for r in range(rows):
        d = np.abs(np.diff(patch[r, :].astype(float)))
        if d.max() < 5:          # skip rows without a clear edge
            continue
        peaks = np.where(d > d.max() * 0.5)[0]
        if len(peaks) > 0:
            edge_pts.append((r, np.average(peaks, weights=d[peaks])))

    if len(edge_pts) < 10:
        return None, None, None, None

    edge_pts = np.array(edge_pts)
    slope, intercept = np.polyfit(edge_pts[:, 0], edge_pts[:, 1], 1)
    cos_a = np.cos(np.arctan(slope))

    # --- Build oversampled ESF ---
    dists, vals = [], []
    for r in range(rows):
        edge_c = slope * r + intercept
        for c in range(cols):
            dists.append((c - edge_c) * cos_a)
            vals.append(patch[r, c])

    dists, vals = np.array(dists), np.array(vals)
    bw = 1.0 / oversample
    bins = np.arange(dists.min(), dists.max(), bw)
    esf = np.zeros(len(bins))
    cnt = np.zeros(len(bins))
    idx = np.clip(((dists - dists.min()) / bw).astype(int), 0, len(bins) - 1)
    for i in range(len(dists)):
        esf[idx[i]] += vals[i]; cnt[idx[i]] += 1
    valid = cnt > 0
    esf[valid] /= cnt[valid]
    esf[~valid] = np.interp(bins[~valid], bins[valid], esf[valid])

    # --- LSF & MTF ---
    lsf = np.diff(gaussian_filter(esf, sigma=0.5))
    lsf_w = lsf * np.hanning(len(lsf))
    mtf = np.abs(np.fft.fft(lsf_w))
    mtf = mtf[:len(mtf) // 2]
    mtf /= mtf[0]
    freq = np.arange(len(mtf)) / (len(mtf) * bw * 2)  # cycles/pixel

    return freq, mtf, esf, lsf

In [ ]:
# Show the selected edge ROI so you can verify / adjust it
r0, r1, c0, c1 = EDGE_ROI_LR

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(lr_ref, cmap='gray')
from matplotlib.patches import Rectangle
axes[0].add_patch(Rectangle((c0, r0), c1-c0, r1-r0,
                             lw=2, edgecolor='red', facecolor='none'))
axes[0].set_title('Full image with edge ROI (red box)')

axes[1].imshow(lr_ref[r0:r1, c0:c1], cmap='gray', interpolation='nearest',
               aspect='auto')
axes[1].set_title(f'Edge ROI ({r1-r0} × {c1-c0} px)')
plt.tight_layout(); plt.show()

In [ ]:
# Measure MTF for every method
roi_hr = tuple(x * f for x in EDGE_ROI_LR)

mtf_curves = {}  # name → (freq_cymm, mtf)

# Native (LR)
fr, mt, _, _ = slanted_edge_mtf(lr_ref, EDGE_ROI_LR, EDGE_ORIENT)
if fr is not None:
    mtf_curves['Native'] = (fr / PITCH_LR_MM, mt)

# SR methods (HR)
for name in ['SAA', 'SAA + IBP', 'TV-SR']:
    fr, mt, _, _ = slanted_edge_mtf(results[name], roi_hr, EDGE_ORIENT)
    if fr is not None:
        mtf_curves[name] = (fr / PITCH_HR_MM, mt)

# ── Plot ──
fig, ax = plt.subplots(figsize=(9, 5))
for name, (freq, mtf) in mtf_curves.items():
    mask = freq <= NYQUIST_HR
    ax.plot(freq[mask], mtf[mask], color=COLORS.get(name, 'gray'),
            lw=1.5, label=name)

ax.axvline(NYQUIST_LR, ls='--', color='gray', alpha=0.6,
           label=f'LR Nyquist ({NYQUIST_LR:.0f} cy/mm)')
ax.axvline(NYQUIST_HR, ls=':', color='gray', alpha=0.6,
           label=f'HR Nyquist ({NYQUIST_HR:.0f} cy/mm)')
ax.axhline(0.5, ls='--', color='orange', alpha=0.4)

ax.set_xlabel('Spatial frequency (cycles/mm)')
ax.set_ylabel('MTF'); ax.set_title('Slanted-Edge MTF')
ax.set_xlim(0, NYQUIST_HR); ax.set_ylim(0, 1.05)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# MTF50 table
print(f'{"Method":>12s}   MTF50 (cy/mm)')
print('-' * 30)
for name, (freq, mtf) in mtf_curves.items():
    above = mtf >= 0.5
    idx = np.where(np.diff(above.astype(int)) == -1)[0]
    if len(idx) > 0:
        i = idx[0]
        f50 = np.interp(0.5, [mtf[i+1], mtf[i]], [freq[i+1], freq[i]])
        print(f'{name:>12s}   {f50:.1f}')
    else:
        print(f'{name:>12s}   —')

### 4b. Siemens Star / Concentric Rings Contrast

The concentric ring pattern in the centre of the ISO 12233 chart has spatial
frequency that **increases** as you move **inward** (smaller radius = finer lines).

At each radius we sample pixel values around the full circumference and compute
**Michelson contrast** = (max − min) / (max + min). Where contrast drops below
~0.1, the system can no longer resolve those features.

In [ ]:
def siemens_contrast(img, center_rc, r_min=5, r_max=120, n_radii=40):
    """Michelson contrast vs radius on concentric rings."""
    cy, cx = center_rc
    radii = np.linspace(r_min, r_max, n_radii)
    angles = np.linspace(0, 2 * np.pi, 360, endpoint=False)
    contrasts = []

    for r in radii:
        ys = (cy + r * np.sin(angles)).astype(int)
        xs = (cx + r * np.cos(angles)).astype(int)
        valid = (ys >= 0) & (ys < img.shape[0]) & (xs >= 0) & (xs < img.shape[1])
        if valid.sum() < 20:
            contrasts.append(0); continue
        vals = img[ys[valid], xs[valid]]
        hi, lo = np.percentile(vals, 95), np.percentile(vals, 5)
        contrasts.append((hi - lo) / max(hi + lo, 1e-6))

    return radii, np.array(contrasts)

In [ ]:
star_curves = {}  # name → (radii_lr, contrast)

# Native
r_n, c_n = siemens_contrast(lr_ref, STAR_CENTER_LR, r_min=5, r_max=120)
star_curves['Native'] = (r_n, c_n)

# SR (at HR coords → convert radii back to LR for comparison)
star_hr = (STAR_CENTER_LR[0] * f, STAR_CENTER_LR[1] * f)
for name in ['SAA', 'SAA + IBP', 'TV-SR']:
    r_s, c_s = siemens_contrast(results[name], star_hr, r_min=10, r_max=240)
    star_curves[name] = (r_s / f, c_s)  # normalise to LR pixel radii

fig, ax = plt.subplots(figsize=(8, 5))
for name, (radii, contrast) in star_curves.items():
    ax.plot(radii, contrast, color=COLORS.get(name, 'gray'), lw=1.5, label=name)
ax.axhline(0.1, ls='--', color='orange', alpha=0.5, label='Contrast = 0.1')
ax.set_xlabel('Radius from centre (LR pixels)  ← higher frequency')
ax.set_ylabel('Michelson contrast')
ax.set_title('Concentric Rings — Contrast vs Radius')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax.invert_xaxis()  # smaller radius = higher freq on the left
plt.tight_layout(); plt.show()

### 4c. Line-Pair Intensity Profiles

A direct, intuitive way to see resolution improvement: extract a horizontal
intensity profile across vertical line pairs of decreasing spacing (from the
ruler-like patterns on the chart sides).

Where the profile oscillations disappear into noise, you've hit the resolution limit.

In [ ]:
row_hr = LP_ROW_LR * f
c0_hr  = LP_COL_RANGE_LR[0] * f
c1_hr  = LP_COL_RANGE_LR[1] * f

fig, axes = plt.subplots(len(results), 1, figsize=(14, 2.2 * len(results)),
                          sharex=True)

for ax, (name, img) in zip(axes, results.items()):
    profile = img[row_hr, c0_hr:c1_hr]
    x_lr = np.arange(len(profile)) / f  # show in LR pixel units
    ax.plot(x_lr, profile, color=COLORS.get(name, 'gray'), lw=0.6)
    ax.set_ylabel(name, fontsize=9, rotation=0, labelpad=70, va='center')
    ax.set_ylim(profile.min() - 10, profile.max() + 10)
    ax.grid(True, alpha=0.15)

axes[-1].set_xlabel('Position (LR pixels)')
fig.suptitle(f'Line-pair profiles at row {LP_ROW_LR}', fontsize=12)
plt.tight_layout(); plt.show()

### 4d. Ground-Truth Comparison (SIFT + ECC → PSNR / SSIM)

Since we know the exact chart design (the ISO 12233 PDF), we can:
1. Rasterize the chart at high resolution
2. Register it to our captured image using **SIFT** (coarse) + **ECC** (fine)
3. Compute **PSNR** and **SSIM** in a central ROI

This tells us how close each SR reconstruction is to the *ideal* image,
capturing both resolution improvement and artefacts.

> **Note:** This section is fragile — the registration depends on having enough
> overlapping features. If it fails, the rest of the notebook still works.

In [ ]:
# Rasterize the chart PDF (only once)
CHART_PNG = 'iso12233_chart.png'
if not os.path.exists(CHART_PNG):
    subprocess.run(
        ['pdftoppm', '-png', '-r', '600', '-singlefile',
         ISO_CHART_PATH, CHART_PNG.replace('.png', '')],
        check=True)
    print('Rasterized chart PDF')

chart_full = load_gray(CHART_PNG)
print(f'Chart image: {chart_full.shape}')

In [ ]:
def sift_align(ref, target):
    """Coarse alignment: SIFT features → homography."""
    to_u8 = lambda x: np.clip(x / max(x.max(), 1) * 255, 0, 255).astype(np.uint8)
    ref_u8, tgt_u8 = to_u8(ref), to_u8(target)

    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(ref_u8, None)
    kp2, des2 = sift.detectAndCompute(tgt_u8, None)
    print(f'  SIFT keypoints: {len(kp1)} / {len(kp2)}')

    if des1 is None or des2 is None or min(len(kp1), len(kp2)) < 4:
        return None, None

    matches = cv2.BFMatcher().knnMatch(des1, des2, k=2)
    good = [m for m, n in matches if m.distance < 0.7 * n.distance]
    print(f'  Good matches: {len(good)}')
    if len(good) < 10:
        return None, None

    src = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(dst, src, cv2.RANSAC, 5.0)
    print(f'  Homography inliers: {mask.ravel().sum()}')

    h, w = ref.shape
    return cv2.warpPerspective(target.astype(np.float32), H, (w, h)), H


def ecc_refine(ref, target, mode='affine'):
    """Fine alignment with Enhanced Correlation Coefficient."""
    to_u8 = lambda x: np.clip(x / max(x.max(), 1) * 255, 0, 255).astype(np.uint8)
    warp_mode = cv2.MOTION_AFFINE if mode == 'affine' else cv2.MOTION_HOMOGRAPHY
    warp_mat  = np.eye(2, 3, dtype=np.float32) if mode == 'affine' \
                else np.eye(3, 3, dtype=np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 200, 1e-6)
    try:
        _, warp_mat = cv2.findTransformECC(
            to_u8(ref), to_u8(target), warp_mat, warp_mode, criteria)
        h, w = ref.shape
        warp_fn = cv2.warpPerspective if mode == 'homography' else cv2.warpAffine
        return warp_fn(target.astype(np.float32), warp_mat, (w, h),
                       flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP), warp_mat
    except cv2.error as e:
        print(f'  ECC failed: {e}')
        return None, None

In [ ]:
# Step 1: Resize chart to roughly match captured image
scale = lr_ref.shape[0] / chart_full.shape[0]
chart_resized = ndi_zoom(chart_full, scale, order=3)
mh = min(chart_resized.shape[0], lr_ref.shape[0])
mw = min(chart_resized.shape[1], lr_ref.shape[1])
chart_crop = chart_resized[:mh, :mw]
ref_crop   = lr_ref[:mh, :mw]
print(f'Resized chart: {chart_crop.shape} vs captured: {ref_crop.shape}')

# Step 2: Coarse (SIFT) → fine (ECC) alignment
print('\nCoarse alignment (SIFT)...')
chart_coarse, H = sift_align(ref_crop, chart_crop)

chart_aligned = None
if chart_coarse is not None:
    print('\nFine alignment (ECC)...')
    chart_fine, _ = ecc_refine(ref_crop, chart_coarse)
    chart_aligned = chart_fine if chart_fine is not None else chart_coarse

    # Intensity normalisation (linear)
    mask = chart_aligned > 10
    if mask.any():
        g = ref_crop[mask].std() / max(chart_aligned[mask].std(), 1e-6)
        chart_aligned = g * chart_aligned + (ref_crop[mask].mean() - g * chart_aligned[mask].mean())

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].imshow(ref_crop, cmap='gray');        axes[0].set_title('Captured')
    axes[1].imshow(chart_aligned, cmap='gray');   axes[1].set_title('Chart (aligned)')
    axes[2].imshow(np.abs(ref_crop - chart_aligned), cmap='hot', vmax=40)
    axes[2].set_title('|Difference|')
    for ax in axes: ax.axis('off')
    plt.tight_layout(); plt.show()
else:
    print('\n⚠ Alignment failed — ground-truth comparison will be skipped.')

In [ ]:
# Step 3: PSNR and SSIM for each method
if chart_aligned is not None:
    chart_hr = ndi_zoom(chart_aligned, f, order=3)
    margin = 50  # LR pixels to discard at borders
    roi = (margin * f, (mh - margin) * f, margin * f, (mw - margin) * f)
    gt = chart_hr[roi[0]:roi[1], roi[2]:roi[3]]

    norm01 = lambda x: (x - x.min()) / max(x.max() - x.min(), 1e-6)
    gt_n = norm01(gt)

    print(f'{"Method":>12s}   {"PSNR (dB)":>10s}   {"SSIM":>6s}')
    print('-' * 35)
    for name, img in results.items():
        crop = img[roi[0]:roi[1], roi[2]:roi[3]]
        sh = min(crop.shape[0], gt_n.shape[0])
        sw = min(crop.shape[1], gt_n.shape[1])
        crop_n = norm01(crop[:sh, :sw])
        gt_c   = gt_n[:sh, :sw]
        p = psnr(gt_c, crop_n)
        s = ssim(gt_c, crop_n, data_range=1.0)
        print(f'{name:>12s}   {p:10.2f}   {s:6.4f}')
else:
    print('Ground-truth comparison skipped (alignment failed).')

---
## 5. Summary Dashboard

Everything on one figure: visual crops, MTF curves, contrast profiles, and
line-pair overlays.

In [ ]:
fig = plt.figure(figsize=(20, 15))
gs = GridSpec(3, 5, figure=fig, hspace=0.30, wspace=0.25)

# ── Row 0: Visual crops of every method ──
for i, (name, img) in enumerate(results.items()):
    ax = fig.add_subplot(gs[0, i])
    ax.imshow(hr_crop(img, cr, cc, 55, f), cmap='gray', interpolation='nearest')
    ax.set_title(name, fontsize=10); ax.axis('off')

# ── Row 1 left: MTF ──
ax_mtf = fig.add_subplot(gs[1, :3])
for name, (freq, mtf) in mtf_curves.items():
    mask = freq <= NYQUIST_HR
    ax_mtf.plot(freq[mask], mtf[mask], color=COLORS.get(name, 'gray'),
                lw=1.5, label=name)
ax_mtf.axvline(NYQUIST_LR, ls='--', color='gray', alpha=0.5,
               label=f'LR Nyquist ({NYQUIST_LR:.0f})')
ax_mtf.axhline(0.5, ls='--', color='orange', alpha=0.4)
ax_mtf.set_xlabel('Spatial freq (cy/mm)')
ax_mtf.set_ylabel('MTF'); ax_mtf.set_title('Slanted-Edge MTF')
ax_mtf.set_xlim(0, NYQUIST_HR); ax_mtf.set_ylim(0, 1.05)
ax_mtf.legend(fontsize=7); ax_mtf.grid(True, alpha=0.3)

# ── Row 1 right: Siemens star contrast ──
ax_star = fig.add_subplot(gs[1, 3:])
for name, (radii, contrast) in star_curves.items():
    ax_star.plot(radii, contrast, color=COLORS.get(name, 'gray'),
                 lw=1.5, label=name)
ax_star.axhline(0.1, ls='--', color='orange', alpha=0.5)
ax_star.set_xlabel('Radius (LR px)  ← higher freq')
ax_star.set_ylabel('Contrast')
ax_star.set_title('Concentric Rings Contrast')
ax_star.set_ylim(0, 1.05); ax_star.invert_xaxis()
ax_star.legend(fontsize=7); ax_star.grid(True, alpha=0.3)

# ── Row 2: Line-pair profiles ──
for i, (name, img) in enumerate(results.items()):
    ax = fig.add_subplot(gs[2, i])
    profile = img[LP_ROW_LR * f, LP_COL_RANGE_LR[0]*f : LP_COL_RANGE_LR[1]*f]
    x = np.arange(len(profile)) / f
    ax.plot(x, profile, color=COLORS.get(name, 'gray'), lw=0.5)
    ax.set_title(name, fontsize=8)
    ax.set_ylim(profile.min() - 10, profile.max() + 10)
    if i == 0: ax.set_ylabel('Intensity')
    ax.set_xlabel('LR px', fontsize=7)
    ax.tick_params(labelsize=6); ax.grid(True, alpha=0.15)

fig.suptitle('Super-Resolution Summary — ISO 12233 Chart',
             fontsize=15, fontweight='bold', y=1.01)
plt.savefig('sr_summary.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: sr_summary.png')

---
## Appendix: Parameter Tuning Guide

| Parameter | Cell | Effect | Start here |
|-----------|------|--------|------------|
| `NOMINAL_SHIFTS` | §0 | XPR-20 shift pattern — **the most important setting**. If SR looks ghosted, permute these. | ±0.5 px, 2×2 grid |
| `USE_MEASURED_SHIFTS` | §0 | Phase-correlation estimates. More accurate if working; can fail at exactly 0.5 px shifts. | `False` |
| `IBP_ITERATIONS` | §0 | More → sharper, but eventually rings | 20–50 |
| `IBP_STEP_SIZE` | §0 | Convergence speed vs stability | 0.3–0.8 |
| `TV_LAMBDA` | §0 | Regularisation strength: larger → smoother, smaller → sharper + noisier | 0.001–0.05 |
| `TV_ITERATIONS` | §0 | Convergence — check the MSE curve | 40–100 |
| `TV_STEP_SIZE` | §0 | Too large → oscillates | 0.05–0.3 |
| `PSF_HALFWIDTH` | §2 | Kernel size = (2×hw+1)². Larger if PSF has wide wings | 3 (→ 7×7) |
| `EDGE_ROI_LR` | §0 | Which edge to measure MTF on — pick a clean, slightly slanted edge | visual |
| `STAR_CENTER_LR` | §0 | Centre of the concentric rings — must be precise | visual |

### Troubleshooting

**SR result looks identical to bicubic upscale:**  
→ Shifts may be wrong. Try `USE_MEASURED_SHIFTS = True`, or manually test all 24 permutations of `NOMINAL_SHIFTS`.

**SR result has visible ghosting / double edges:**  
→ Shift magnitudes are off. Try scaling the nominal shifts by 0.8–1.2.

**IBP diverges (MSE increases):**  
→ Reduce `IBP_STEP_SIZE` to 0.2 or lower.

**TV-SR is too smooth:**  
→ Reduce `TV_LAMBDA` by 2–5×.

**Ground-truth alignment fails:**  
→ Your crop may not overlap enough with the full chart. Try imaging a larger portion, or skip §4d.